# Example 11.1 / 6: Check Convexity Component by Component

Companion notebook for **11.1 Quadratic Models** in the AMPL lab Documentation, section "Check Convexity Component by Component". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Check Convexity Component by Component

For a continuous model, check each component's curvature against its role:

| Component | Convex model condition | Example | Result |
|---|---|---|---|
| minimized quadratic objective | Hessian positive semidefinite | `minimize x^2 - 2*x` | Hessian $2\ge0$: convex, minimizer $x=1$ |
| maximized quadratic objective | Hessian negative semidefinite | `maximize 4*x - x^2` | Hessian $-2\le0$: concave, maximizer $x=2$ |
| constraint `quadratic <= affine` | left-minus-right is convex | `x^2 + y^2 <= 4` | left minus right is convex: a disk, a convex set |
| constraint `quadratic >= affine` | left-minus-right is concave | `-x^2 >= -4` | left minus right $4-x^2$ is concave: the interval $[-2,2]$ |
| quadratic equality | generally nonconvex unless it reduces to an affine equality | `x^2 = 1` | the set $\{-1,1\}$; its midpoint $0$ is excluded, so it is nonconvex |

Positive *definiteness* is stronger than positive semidefiniteness. It gives strict convexity
of that quadratic over all directions, but uniqueness still concerns the feasible directions
and the complete objective. A semidefinite objective may have many optimal points.

The positive semidefinite objective $(x+y-2)^2$ on $0\le x,y\le2$ is flat along $x+y=2$. Ipopt
from two starts returns two different optimal points with the same objective value $0$:

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
var x >= 0, <= 2;  # first coordinate, box [0, 2]
var y >= 0, <= 2;  # second coordinate, box [0, 2]

minimize Flat: (x + y - 2)^2;  # PSD objective, flat along x + y = 2

Writing model.mod


### amplpy: run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# run.py
for start in ((0, 0), (2, 0.5)):
    ampl.var["x"].set_value(start[0])
    ampl.var["y"].set_value(start[1])
    ampl.solve(solver="ipopt", verbose=False)
    print(start, ampl.solve_result, round(ampl.var["x"].value(), 4),
          round(ampl.var["y"].value(), 4), round(ampl.obj["Flat"].value(), 8))

(0, 0) solved 1.0 1.0 0.0
(2, 0.5) solved 1.4916 0.5084 0.0
